# One-head GATv2 tensor-v2 data-scaling curve

## Goal

Train the same one-head, three-layer GATv2 at nested **25%, 50%, 100%, and
200%** training scales while freezing the original validation, test, and
exemplar cohorts.

- 25% and 50% are deterministic, family-stratified `group_id` subsets of the
  original four archive cohorts.
- 100% is all official training groups in archive cohorts 1–4.
- 200% is all official training groups in archive cohorts 1–8.
- Validation, test, and exemplar always use archive cohorts 1–4.

This makes the subsets nested without splitting sibling designs and prevents
new 200% data from changing the evaluation distribution.

The notebook is resumable within and across Kaggle sessions. Each scale has its
own optimizer backup, best checkpoint, result directory, and provenance
signature. Results are repackaged after every run.

## Suggested session split

- Session 1: `ACTIVE_SCALES = [25, 50, 100]` (default; 10 hours of training caps).
- Session 2: attach the first result archive, set `PREVIOUS_RESULTS_ROOT`, and
  use `ACTIVE_SCALES = [200]` (10-hour cap).

Normal early stopping can finish any scale before its cap. Both sessions must use the same final eight-archive pinned tensor revision.


In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ll-hls4ml"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/ll-hls4ml.git"
# Fill this after committing/pushing the scaling code.
REPO_REF = "REPLACE_WITH_EXACT_LL_HLS4ML_COMMIT"

TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors"
# Required: use the immutable Hugging Face dataset commit, never "main".
TENSOR_REVISION = "REPLACE_WITH_EXACT_HF_DATASET_COMMIT"
HF_CACHE_DIR = Path("/tmp/wa_hls4ml_hf_cache")

# Attach a prior Kaggle result archive and point this at its input directory to
# resume an interrupted scale or collect earlier completed scale results.
PREVIOUS_RESULTS_ROOT = None

SEED = 42
TRAIN_SUBSET_SEED = 42
BASELINE_ARCHIVES_PER_FAMILY = 4
ACTIVE_SCALES = [25, 50, 100]  # Second session: [200]
SCALE_TRAIN_BUDGETS = {
    25: "120m",
    50: "180m",
    100: "300m",
    200: "600m",
}
ARCHIVE_NAME = "ll_hls4ml_gatv2_scaling_results"


def assert_commit(value, name):
    assert re.fullmatch(r"[0-9a-fA-F]{40,64}", value), (
        f"{name} must be an immutable full commit hash; got {value!r}"
    )


assert_commit(REPO_REF, "REPO_REF")
assert_commit(TENSOR_REVISION, "TENSOR_REVISION")
assert set(ACTIVE_SCALES) <= {25, 50, 100, 200}

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator before running."
print("PyTorch:", torch.__version__)
print("CUDA devices:", GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))
print("Active scales:", ACTIVE_SCALES)


## Install dependencies and checkout immutable training code

In [ ]:
# Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q torch-geometric huggingface_hub pyyaml pandas matplotlib


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", REPO_REF],
    check=True,
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
assert commit == REPO_REF, (commit, REPO_REF)
print("ll-hls4ml commit:", commit)

sys.path.insert(0, str(REPO_DIR / "src"))
from ll_hls4ml.data.tensorize import EMBED_SIZE
from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE,
    GRAPH_CONTEXT_CATEGORICAL_VOCABS,
    GRAPH_CONTEXT_NUMERIC_KEYS,
    PRAGMA_FEATURE_SIZE,
)
from ll_hls4ml.models.registry import list_models

assert "hetero_gat" in list_models()


## Download the pinned tensor dataset

In [ ]:
from huggingface_hub import login, snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Cache disk usage:")
subprocess.run(["df", "-h", "/tmp"], check=True)

print("Downloading", TENSOR_REPO_ID, "at", TENSOR_REVISION)
TENSOR_DIR = Path(
    snapshot_download(
        repo_id=TENSOR_REPO_ID,
        repo_type="dataset",
        revision=TENSOR_REVISION,
        token=hf_token,
        cache_dir=str(HF_CACHE_DIR),
        allow_patterns=["*.pt", "*.json"],
    )
)
print("Tensor snapshot downloaded to", TENSOR_DIR)


## Validate tensor-v2 and archive availability

In [ ]:
tensor_files = sorted(TENSOR_DIR.rglob("*.pt"))
labels_path = TENSOR_DIR / "labels.json"
assert labels_path.is_file(), "tensor-v2 labels.json is missing"
labels_payload = json.loads(labels_path.read_text())
label_count = len(labels_payload.get("labels", {}))
assert tensor_files, f"No .pt tensors found under {TENSOR_DIR}"

sample = torch.load(tensor_files[0], map_location="cpu", weights_only=False)
errors = []
for node_type in ("variable", "constant"):
    width = sample[node_type].x.shape[1]
    if width != EMBED_SIZE:
        errors.append(f"{node_type} width {width} != {EMBED_SIZE}")
if sample["pragma"].x.shape[1] != PRAGMA_FEATURE_SIZE:
    errors.append(
        f"pragma width {sample['pragma'].x.shape[1]} != {PRAGMA_FEATURE_SIZE}"
    )
if sample["block"].x.shape[1] != BLOCK_FEATURE_SIZE:
    errors.append(
        f"block width {sample['block'].x.shape[1]} != {BLOCK_FEATURE_SIZE}"
    )
if not hasattr(sample, "graph_context_categorical"):
    errors.append("graph_context_categorical is absent")
elif sample.graph_context_categorical.shape[-1] != len(
    GRAPH_CONTEXT_CATEGORICAL_VOCABS
):
    errors.append("categorical synthesis-context width is incorrect")
if not hasattr(sample, "graph_context_numeric"):
    errors.append("graph_context_numeric is absent")
elif sample.graph_context_numeric.shape[-1] != len(
    GRAPH_CONTEXT_NUMERIC_KEYS
):
    errors.append("numeric synthesis-context width is incorrect")
assert not errors, "Downloaded tensors are not tensor-v2:\n- " + "\n- ".join(errors)

vocab_candidates = [
    TENSOR_DIR / "vocab.json",
    REPO_DIR / "artifacts" / "vocab" / "vocab.json",
]
VOCAB_PATH = next((path for path in vocab_candidates if path.is_file()), None)
assert VOCAB_PATH is not None, "vocab.json is missing"

KERNEL_TYPES = [
    "2layer",
    "3layer",
    "conv1d",
    "conv2d",
    "dense_latency",
    "dense_resource",
    "rule4ml",
]


def natural_key(value):
    return [
        int(part) if part.isdigit() else part
        for part in re.split(r"(\d+)", value)
    ]


archive_inventory = {
    family: sorted(
        [
            path.name
            for path in (TENSOR_DIR / family).iterdir()
            if path.is_dir()
        ],
        key=natural_key,
    )
    for family in [*KERNEL_TYPES, "exemplar"]
}
for family, archives in archive_inventory.items():
    print(f"{family:16s}", len(archives), archives)

required_training_archives = 2 * BASELINE_ARCHIVES_PER_FAMILY
for family in KERNEL_TYPES:
    assert len(archive_inventory[family]) >= required_training_archives, (
        f"The complete scaling snapshot requires {required_training_archives} "
        f"archive cohorts for {family}; found {archive_inventory[family]}"
    )
assert len(archive_inventory["exemplar"]) >= BASELINE_ARCHIVES_PER_FAMILY

print("Tensor files:", len(tensor_files))
print("Indexed labels:", label_count)
print("Vocabulary:", VOCAB_PATH)


## Matched scaling configurations

Only the training-set scale changes. All models retain seed 42, one attention
head, three layers, hidden width 64, batch size one per GPU, patience 30,
multi-pooling, the global shortcut, split resource/timing heads, DSP/BRAM hurdle
heads, and the log-Huber hurdle loss.

The trainer writes `data_scale_manifest.json`, `split_manifest.json`, and the
exact tensor source revision into every result.


In [ ]:
common = {
    "model": "hetero_gat",
    "tensor_dir": str(TENSOR_DIR),
    "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH),
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "seed": SEED,
    "train_subset_seed": TRAIN_SUBSET_SEED,
    "baseline_archives_per_family": BASELINE_ARCHIVES_PER_FAMILY,
    "evaluation_archives_per_family": BASELINE_ARCHIVES_PER_FAMILY,
    "strict_archive_counts": True,
    "split_strategy": "official_or_stratified",
    "family_balanced_sampling": False,
    "batch_size": 1,
    "num_workers": 0,
    "epochs": 400,
    "patience": 30,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "heads": 1,
    "dropout": 0.15,
    "pool": "multi",
    "aggr": "sum",
    "use_global_features": True,
    "use_context": False,
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "verbose": 2,
}

scale_configs = {}
for scale in (25, 50, 100, 200):
    experiment = f"kaggle_v2_gatv2_1head_scale{scale:03d}_seed42"
    scale_configs[scale] = {
        **common,
        "experiment_name": experiment,
        "checkpoint_dir": str(RESULTS_DIR / experiment / "checkpoints"),
        "train_scale": scale / 100,
        "scale_percent": scale,
    }

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
_cached_previous_root = None


def previous_search_root():
    global _cached_previous_root
    if PREVIOUS_RESULTS_ROOT is None:
        return None
    if _cached_previous_root is not None:
        return _cached_previous_root
    root = Path(PREVIOUS_RESULTS_ROOT)
    archives = sorted(root.rglob(f"{ARCHIVE_NAME}.zip"))
    if archives:
        if len(archives) > 1:
            raise RuntimeError(f"Multiple previous result archives: {archives}")
        extracted = WORK / "previous_scaling_results"
        if not extracted.is_dir():
            shutil.unpack_archive(archives[0], extracted)
            print("Extracted previous results:", archives[0])
        root = extracted
    _cached_previous_root = root
    return root


def resume_signature(config):
    keys = [
        "model",
        "seed",
        "train_subset_seed",
        "train_scale",
        "baseline_archives_per_family",
        "evaluation_archives_per_family",
        "hidden_dim",
        "num_layers",
        "heads",
        "dropout",
        "pool",
        "aggr",
        "use_global_features",
        "use_context",
        "split_heads",
        "hurdle_heads",
        "loss",
        "learning_rate",
        "weight_decay",
        "batch_size",
    ]
    return {
        **{key: config.get(key) for key in keys},
        "repo_ref": REPO_REF,
        "tensor_source_revision": TENSOR_REVISION,
    }


def find_previous_run(experiment):
    root = previous_search_root()
    if root is None:
        return None
    matches = sorted(
        path.parent
        for path in root.rglob("notebook_resume_signature.json")
        if path.parent.name == experiment
    )
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple previous runs found for {experiment}: {matches}"
        )
    return matches[0] if matches else None


def prepare_run(config):
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    signature_path = run_dir / "notebook_resume_signature.json"
    expected_signature = resume_signature(config)

    previous_run = find_previous_run(experiment)
    if not run_dir.exists() and previous_run is not None:
        previous_signature = json.loads(
            (previous_run / "notebook_resume_signature.json").read_text()
        )
        assert previous_signature == expected_signature, (
            f"Refusing incompatible resume for {experiment}"
        )
        shutil.copytree(previous_run, run_dir)
        print("Imported previous run:", previous_run)

    run_dir.mkdir(parents=True, exist_ok=True)
    if signature_path.is_file():
        existing_signature = json.loads(signature_path.read_text())
        assert existing_signature == expected_signature, (
            f"Local run provenance changed for {experiment}; use a new "
            "experiment name or remove the incompatible local run."
        )
    signature_path.write_text(json.dumps(expected_signature, indent=2))

    checkpoint_dir = Path(config["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    backup = checkpoint_dir / f"{experiment}_backup.pt"
    payload = dict(config)
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
        print("Will resume", experiment, "from", backup)

    config_path = CONFIG_DIR / f"{experiment}.json"
    config_path.write_text(json.dumps(payload, indent=2))
    return config_path


for scale, config in scale_configs.items():
    prepare_run(config)


## Time-bounded training and incremental packaging

In [ ]:
import shlex
import time

TRAIN_SCRIPT = REPO_DIR / "scripts" / "train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"


def base_training_command(config_path):
    if GPU_COUNT > 1:
        return [
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--standalone",
            f"--nproc_per_node={GPU_COUNT}",
            str(TRAIN_SCRIPT),
            "--config",
            str(config_path),
        ]
    return [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--config",
        str(config_path),
    ]


def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=run_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()


def package_results():
    archive = Path(
        shutil.make_archive(
            str(WORK / ARCHIVE_NAME),
            "zip",
            root_dir=RESULTS_DIR,
        )
    )
    print("Updated result archive:", archive)
    return archive


def run_scale(scale):
    if scale not in ACTIVE_SCALES:
        print(f"Skipping {scale}% by ACTIVE_SCALES configuration.")
        return

    config = scale_configs[scale]
    config_path = prepare_run(config)
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"

    if summary_path.is_file():
        existing = json.loads(summary_path.read_text())
        evaluation_checkpoint = existing.get("resolved_config", {}).get(
            "evaluation_checkpoint_path"
        )
        if evaluation_checkpoint is None:
            print(experiment, "already completed normally; skipping.")
            package_results()
            return
        print(experiment, "has a timeout evaluation; resuming training.")

    command = [
        "timeout",
        "--signal=INT",
        "--kill-after=5m",
        SCALE_TRAIN_BUDGETS[scale],
        *base_training_command(config_path),
    ]

    try:
        started = time.time()
        return_code = run_and_stream(
            command,
            run_dir / "training.log",
        )
        elapsed = time.time() - started
        print(experiment, "training return code:", return_code)
        print(experiment, "training wall seconds:", round(elapsed, 1))

        if return_code == 0 and summary_path.is_file():
            print("Training completed normally; evaluation is already complete.")
            return

        best_checkpoint = (
            Path(config["checkpoint_dir"])
            / f"{experiment}_checkpoint.pt"
        )
        assert best_checkpoint.is_file(), (
            f"No best checkpoint exists for {experiment}: {best_checkpoint}"
        )
        print("Evaluating best checkpoint:", best_checkpoint)
        evaluation_command = [
            sys.executable,
            str(TRAIN_SCRIPT),
            "--config",
            str(config_path),
            "--evaluate-checkpoint",
            str(best_checkpoint),
        ]
        evaluation_code = run_and_stream(
            evaluation_command,
            run_dir / "evaluation.log",
        )
        assert evaluation_code == 0, (
            f"Best-checkpoint evaluation failed with code {evaluation_code}"
        )
    finally:
        package_results()


## Run 25%

In [ ]:
run_scale(25)


## Run 50%

In [ ]:
run_scale(50)


## Run 100%

In [ ]:
run_scale(100)


## Run 200%

For the recommended two-session schedule this prints a skip in session 1. In
session 2, attach the first archive, set `PREVIOUS_RESULTS_ROOT`, and change
`ACTIVE_SCALES` to `[200]`.


In [ ]:
run_scale(200)


## Compare the available curve and download results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for scale, config in scale_configs.items():
    run_dir = RESULTS_DIR / config["experiment_name"]
    metrics_path = run_dir / "metrics.csv"
    summary_path = run_dir / "summary.json"
    scale_manifest_path = run_dir / "data_scale_manifest.json"
    if not (
        metrics_path.is_file()
        and summary_path.is_file()
        and scale_manifest_path.is_file()
    ):
        continue

    metrics = pd.read_csv(metrics_path)
    metrics = metrics[metrics["kernel_family"] == "all"]
    summary = json.loads(summary_path.read_text())
    scale_manifest = json.loads(scale_manifest_path.read_text())
    for split in ("test", "exemplar"):
        selected = metrics[metrics["split"] == split]
        rows.append(
            {
                "scale_percent": scale,
                "actual_train_samples": summary["sizes"]["train"],
                "split": split,
                "macro_smape": selected["smape"].mean(),
                "macro_r2": selected["r2"].mean(),
                "best_epoch": summary.get("best_epoch"),
                "best_validation_smape": summary.get("best_metric"),
                "tensor_source_revision": summary["resolved_config"].get(
                    "tensor_source_revision"
                ),
            }
        )

comparison = pd.DataFrame(rows).sort_values(
    ["split", "actual_train_samples"]
)
display(comparison)
comparison.to_csv(RESULTS_DIR / "scaling_summary.csv", index=False)

test_curve = comparison[comparison["split"] == "test"]
if not test_curve.empty:
    figure, axis = plt.subplots(figsize=(7, 4.5))
    axis.plot(
        test_curve["actual_train_samples"],
        test_curve["macro_smape"],
        marker="o",
        linewidth=2,
    )
    for row in test_curve.itertuples():
        axis.annotate(
            f"{row.scale_percent}%",
            (row.actual_train_samples, row.macro_smape),
            xytext=(5, 5),
            textcoords="offset points",
        )
    axis.set_xscale("log", base=2)
    axis.set_xlabel("Actual training graphs")
    axis.set_ylabel("Test macro SMAPE (%)")
    axis.set_title("One-head GATv2 data-scaling curve")
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(
        RESULTS_DIR / "scaling_curve.png",
        dpi=180,
        bbox_inches="tight",
    )
    display(figure)
    plt.close(figure)

archive = package_results()
from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation guardrails

- Use **actual training graph count**, not the nominal percentage, on the
  learning-curve x-axis.
- The curve is one training seed. Treat changes below roughly two SMAPE points
  cautiously until replicated.
- A time-capped point is right-censored. Inspect its best epoch and learning
  curve before calling it saturated.
- Compare per-family and per-target metrics. Overall improvement accompanied by
  flat Rule4ML performance means additional IID data is not fixing the dominant
  hard domain.
- Do not compare or resume runs unless the immutable tensor source revision
  agrees.
